# Processo Seletivo: Engenharia de Dados – TDW BI Consulting
**Candidato:** Eduardo de Castro Bello  
**Notebook v3:** Pipeline PySpark com Boas Práticas, Schemas Estritos e Arquitetura Lakehouse



**Data:** 27/10/2026


In [ ]:
## Registro de correções: v2 → v3

Após entregar a v2, revisei o notebook e encontrei erros. Esta versão corrige os pontos abaixo (a lógica das Q1, Q2, Q3 e Q5 já estava correta na v2 e foi mantida).

| Etapa | Problema na v2 | Correção na v3 |
| :--- | :--- | :--- |
| **Q4 (últimos 6 meses)** | **Não havia filtro de período**: o ranking considerava todo o período. A coluna `date` ficava como string (`dd/MM/yyyy`). | `date` convertida para o tipo data; filtro da janela de 6 meses terminando na maior data de `sales.csv`. |
| **Q7 (Parquet)** | Erro de digitação no caminho de saída (`"output / parquet"`): o arquivo era gravado em pastas com espaço no nome, fora de `output/parquet`. | Gravação em `output/parquet`. Desempate do ranking por `product_id` para resultado determinístico. |
| **Schema de users** | Schema declarado como `id, name, email, signup_date`, mas o CSV tem `client_id, name, email, gender, login, password`: a coluna `signup_date` continha o gênero. | Schema alinhado ao arquivo (`client_id, name, email, gender`); `login` e `password` não são carregadas. Joins passam a usar `client_id`. |
| **Silver (ETL)** | Duas etapas de ETL redundantes (uma com `inferSchema`, outra com schema estrito). | Uma única etapa com schema estrito e conversão de `date` e `price`. |
| **Validações** | Apenas impressões de nulos e órfãos, sem ação. | `assert` para duplicidade de `sale_id`, chaves nulas e produtos órfãos; conferência de volumetria (Q1) e do total por cliente (Q2). |
| **Q2 / Q5 / Q6** | Total por cliente arredondado antes das estatísticas. | Valor sem arredondamento nas etapas intermediárias; arredondamento só na exibição (Q3, Q5). |
| **Q6** | Limites de gold/silver com sobreposição na fronteira e `show(500)`. | Limites sem sobreposição (gold: `> média - desvio`; silver: `<=`) e contagem por categoria. |
| **Organização** | Q3 duplicada, células sobrando (`products2.json`, célula vazia) e texto de apresentação extenso. | Removidos; notebook reduzido ao necessário. |

**Premissas adotadas:** valor final da compra = `qtde * price`; "últimos seis meses" = período dos dados disponíveis (até a maior data de `sales.csv`); desvio padrão amostral (`F.stddev`).


---
**Validações:** antes das análises, o pipeline checa duplicidade de `sale_id`, chaves nulas e produtos órfãos nas vendas, e confere a volumetria (Q1) e a soma por cliente contra o total geral (Q2).

**Limitações conhecidas:** o `products.json` é corrigido no próprio arquivo da camada raw (o ideal seria gravar uma cópia em outra camada), e não há log em arquivo.

**Uso de IA:** a revisão e as correções da v3 foram feitas com apoio de IA (uso liberado pelo recrutador); revisei o resultado e executei o notebook completo.


In [ ]:
#from google.colab import drive
#drive.mount('/content/drive')

In [13]:
# ==================================================
# [SETUP] ESTRUTURA DE DIRETÓRIOS E INSTÂNCIA SPARK
# ==================================================
import json, os, urllib.request
from pathlib import Path
from pyspark.sql import SparkSession, Window
from pyspark.sql import functions as F
from pyspark.sql.types import StructType, StructField, StringType, IntegerType, DoubleType

workspace = Path("/content/lakehouse_workspace")
raw = workspace / "data" / "raw"
processed = workspace / "data" / "processed"
output = workspace / "output" / "parquet"
for d in (raw, processed, output):
    d.mkdir(parents=True, exist_ok=True)

spark = SparkSession.builder.appName('teste_edubello_v3').getOrCreate()
print("[INFO] Sessão Spark e Diretórios criados com sucesso!")

[INFO] Sessão Spark e Diretórios criados com sucesso!


In [14]:
# ==================================================
# [INGESTÃO & NORMALIZAÇÃO RAW]
# ==================================================
path_products = str(raw / "products.json")
path_sales = str(raw / "sales.csv")
path_users = str(raw / "users.csv")

BASE_URL = "https://raw.githubusercontent.com/KaelDucatti/pyspark_test_data/main"
urllib.request.urlretrieve(f"{BASE_URL}/products.json", path_products)
urllib.request.urlretrieve(f"{BASE_URL}/sales.csv", path_sales)
urllib.request.urlretrieve(f"{BASE_URL}/users.csv", path_users)

# Correção do JSON Malformado adicionando colchetes de array
content = Path(path_products).read_text(encoding="utf-8").strip().rstrip(",")
Path(path_products).write_text("[" + content + "]", encoding="utf-8")
print("[INFO] Ingestão e normalização concluídas!")

[INFO] Ingestão e normalização concluídas!


In [15]:
# ==================================================
# [CAMADA SILVER] SCHEMAS ESTRICTOS E PARQUET
# ==================================================
schema_users = StructType([
    StructField("client_id", IntegerType(), True),
    StructField("name", StringType(), True),
    StructField("email", StringType(), True),
    StructField("gender", StringType(), True)
])

schema_sales = StructType([
    StructField("sale_id", IntegerType(), True),
    StructField("date", StringType(), True),
    StructField("product_id", IntegerType(), True),
    StructField("client_id", IntegerType(), True),
    StructField("qtde", IntegerType(), True)
])

schema_products = StructType([
    StructField("product_id", IntegerType(), True),
    StructField("product", StringType(), True),
    StructField("price", StringType(), True)
])

df_users = spark.read.option("header", "true").schema(schema_users).csv(path_users)
df_sales = spark.read.option("header", "true").schema(schema_sales).csv(path_sales) \
    .withColumn("date", F.to_date("date", "dd/MM/yyyy"))
df_products = spark.read.option("multiLine", "true").schema(schema_products).json(path_products) \
    .withColumn("price", F.regexp_replace("price", r"[^0-9.]", "").cast(DoubleType()))

df_users.write.mode("overwrite").parquet(str(processed / "users"))
df_sales.write.mode("overwrite").parquet(str(processed / "sales"))
df_products.write.mode("overwrite").parquet(str(processed / "products"))

df_users = spark.read.parquet(str(processed / "users"))
df_sales = spark.read.parquet(str(processed / "sales"))
df_products = spark.read.parquet(str(processed / "products"))
print("[INFO] Dados limpos e gravados na camada Silver!")

[INFO] Dados limpos e gravados na camada Silver!


In [16]:
# ==================================================
# [DATA QUALITY GATES - ASSERT]
# ==================================================
n_sales = df_sales.count()
dup_sales = n_sales - df_sales.select("sale_id").distinct().count()
null_keys = df_sales.filter(F.col("product_id").isNull() | F.col("client_id").isNull()).count()
orphan_products = df_sales.join(df_products, "product_id", "left_anti").count()

assert dup_sales == 0 and null_keys == 0, "Erro: Vendas contêm duplicatas ou chaves nulas!"
assert orphan_products == 0, "Erro: Existem produtos órfãos nas vendas!"
print("[INFO] Validações de qualidade aprovadas com sucesso!")

[INFO] Validações de qualidade aprovadas com sucesso!


## Resolução das Questões do Teste

In [17]:
# [Q1] Nome do produto e valor final da compra
df_sales_enriched = df_sales.join(F.broadcast(df_products), "product_id", "inner") \
    .withColumn("valor_final_compra", F.col("qtde") * F.col("price"))

df_q1 = df_sales_enriched.select(F.col("product").alias("nome_produto"), "valor_final_compra")
assert df_sales_enriched.count() == n_sales, "Validação Q1: Linhas divergentes pós-join"
df_q1.show(5, truncate=False)

+------------------+------------------+
|nome_produto      |valor_final_compra|
+------------------+------------------+
|Blusa Cropped     |214.5             |
|Camiseta Estampada|218.38            |
|Óculos de Sol     |244.54            |
|Bolsa Tote        |105.96            |
|Camisa Social     |161.52            |
+------------------+------------------+
only showing top 5 rows


In [18]:
# [Q2] Valor total gasto por cliente
df_q2 = df_sales_enriched.groupBy("client_id") \
    .agg(F.sum("valor_final_compra").alias("total_gasto"))

tot_q1 = df_q1.agg(F.sum("valor_final_compra")).first()[0]
tot_q2 = df_q2.agg(F.sum("total_gasto")).first()[0]
assert abs(tot_q1 - tot_q2) < 1e-6, "Validação Q2: Soma por cliente difere do total geral"
df_q2.orderBy(F.col("total_gasto").desc()).show(5, truncate=False)

+---------+------------------+
|client_id|total_gasto       |
+---------+------------------+
|131      |2240.51           |
|245      |1902.85           |
|24       |1843.35           |
|200      |1795.8600000000001|
|31       |1755.6            |
+---------+------------------+
only showing top 5 rows


In [19]:
# [Q3] Top 5 clientes: nome, e-mail e valor gasto
df_q3 = df_q2.join(F.broadcast(df_users), "client_id", "inner") \
    .select(F.col("name").alias("nome_cliente"), "email", F.round("total_gasto", 2).alias("total_gasto")) \
    .orderBy(F.col("total_gasto").desc(), "email") \
    .limit(5)
df_q3.show(truncate=False)

+-----------------+-------------------------+-----------+
|nome_cliente     |email                    |total_gasto|
+-----------------+-------------------------+-----------+
|Randa Friedenbach|rfriedenbach3m@paypal.com|2240.51    |
|Giuditta Blease  |gblease6s@friendfeed.com |1902.85    |
|Cher Higford     |chigfordn@issuu.com      |1843.35    |
|Keen Juggings    |kjuggings5j@phoca.cz     |1795.86    |
|Alfie Pattlel    |apattlelu@discuz.net     |1755.6     |
+-----------------+-------------------------+-----------+



In [20]:
# [Q4] Top 5 produtos nos últimos 6 meses
max_date = df_sales.agg(F.max("date")).first()[0]
cutoff = df_sales.agg(F.add_months(F.max("date"), -6)).first()[0]

df_q4 = df_sales_enriched.filter(F.col("date") > F.lit(cutoff)) \
    .groupBy("product_id", F.col("product").alias("nome_produto")) \
    .agg(F.sum("qtde").alias("quantidade_vendida")) \
    .orderBy(F.col("quantidade_vendida").desc(), "product_id") \
    .limit(5) \
    .select("nome_produto", "quantidade_vendida")
df_q4.show(truncate=False)

+--------------------+------------------+
|nome_produto        |quantidade_vendida|
+--------------------+------------------+
|Macaquinho          |39                |
|Blusa de Manga Longa|34                |
|Casaco de Inverno   |29                |
|Blusa de Malha      |29                |
|Bolsa de Couro      |27                |
+--------------------+------------------+



In [21]:
# [Q5] Média e desvio padrão do faturamento por cliente
df_q5 = df_q2.agg(
    F.round(F.avg("total_gasto"), 2).alias("media_faturamento_cliente"),
    F.round(F.stddev("total_gasto"), 2).alias("desvio_padrao_faturamento")
)
df_q5.show(truncate=False)

stats = df_q2.agg(F.avg("total_gasto").alias("m"), F.stddev("total_gasto").alias("s")).first()
media, desvio = stats["m"], stats["s"]

+-------------------------+-------------------------+
|media_faturamento_cliente|desvio_padrao_faturamento|
+-------------------------+-------------------------+
|707.26                   |414.34                   |
+-------------------------+-------------------------+



In [22]:
# [Q6] Classificação dos Clientes (Silver, Gold, Platinum)
df_q6 = df_q2.withColumn(
    "categoria",
    F.when(F.col("total_gasto") > media, "platinum")
     .when(F.col("total_gasto") > (media - desvio), "gold")
     .otherwise("silver")
)
df_q6.groupBy("categoria").count().show()

+---------+-----+
|categoria|count|
+---------+-----+
|   silver|   50|
|     gold|  108|
| platinum|  133|
+---------+-----+



In [23]:
# [Q7] Top 3 produtos por categoria de cliente e exportação Parquet
df_consumo = df_sales_enriched \
    .join(df_q6.select("client_id", "categoria"), "client_id", "inner") \
    .groupBy("categoria", "product_id", F.col("product").alias("nome_produto")) \
    .agg(F.sum("qtde").alias("total_consumido"))

w = Window.partitionBy("categoria").orderBy(F.col("total_consumido").desc(), F.col("product_id").asc())
df_q7 = df_consumo.withColumn("ranking", F.row_number().over(w)) \
    .filter("ranking <= 3") \
    .orderBy("categoria", "ranking")

df_q7.coalesce(1).write.mode("overwrite").parquet(str(output))
print(f"[SUCESSO] Parquet gerado com sucesso em: {output}")
df_q7.show(truncate=False)

[SUCESSO] Parquet gerado com sucesso em: /content/lakehouse_workspace/output/parquet
+---------+----------+---------------------+---------------+-------+
|categoria|product_id|nome_produto         |total_consumido|ranking|
+---------+----------+---------------------+---------------+-------+
|gold     |12        |Casaco de Inverno    |26             |1      |
|gold     |18        |Cinto de Couro       |26             |2      |
|gold     |34        |Camisa Xadrez        |23             |3      |
|platinum |7         |Blusa de Manga Longa |57             |1      |
|platinum |44        |Mochila de Lona      |52             |2      |
|platinum |49        |Calça Cintura Alta   |45             |3      |
|silver   |39        |Macaquinho           |7              |1      |
|silver   |8         |Sapatos de Salto Alto|6              |2      |
|silver   |2         |Calça Jeans          |5              |3      |
+---------+----------+---------------------+---------------+-------+



---
### Entrega
Execute a célula abaixo **por último**, depois de rodar todo o notebook, para gerar o `historico_exec.txt` (conforme instruções do teste).


In [ ]:
# ==================================================
# [ENTREGA] GERACAO DO HISTORICO DE EXECUCAO
# ==================================================

with open('/content/historico_exec.txt', 'w') as f:
    for i, cmd in enumerate(In):
        if cmd.strip():
            f.write(f"\n\nIn[{i}]:\n{cmd}\n\n")

# Baixando o arquivo gerado
from google.colab import files
files.download('/content/historico_exec.txt')